In [48]:
import pandas as pd
import numpy as np
from tqdm import tqdm

tqdm.pandas()

In [2]:
df = pd.read_excel("/Users/rhombus19/projects/eleo/Unterlagen Teilnehmer/auftraege_mit_sku_Teilnehmer.xlsx")

In [3]:
# 1. SKU
# 2. Order Timestamp
# 3. Anzahl
# 4. Wetter
# 5. Aktionen

In [4]:
df = df.rename(columns={
    "Auftragspositionen/Menge": "qty",
    "Gesamtpreis €": "total_price",
    "Auftragsdatum": "order_time"
})


In [5]:
# Fix SKU formatting
df["SKU"] = df["SKU"].str.replace(r'.', "").str.lower()

In [6]:
all_skus = df["SKU"].unique().tolist()

In [7]:
df["price_per_unit"] = df["total_price"] / df["qty"]

In [8]:
df

,order_time,qty,total_price,SKU,price_per_unit
0,2025-05-26 06:30:18,1,312.68,ga10333,312.680
1,2024-09-23 12:21:21,2,450.37,ga10161,225.185
2,2024-10-09 12:18:14,1,224.57,ga10161,224.570
3,2024-09-19 12:29:59,1,205.70,ga10312,205.700
4,2025-06-18 14:59:13,1,298.15,ga10313,298.150
...,...,...,...,...,...
6352,2024-08-13 08:42:00,3,112.50,sw10004,37.500
6353,2024-07-15 11:25:24,1,50.67,sw100171,50.670
6354,2024-08-13 08:42:00,2,75.00,sw10017,37.500
6355,2024-06-17 07:26:08,1,50.42,sw100173,50.420


In [9]:
mean_price_per_sku = df.groupby("SKU")["price_per_unit"].mean()

In [10]:
mean_price_per_sku

SKU
9101dx     1295.228583
9101fz     1119.547500
9101sf     1468.438148
9101ub      864.650000
9102dx     1937.436548
              ...     
sz21011      74.790000
sz21012      74.375417
sz21022      59.830000
sz21031     108.120556
sz21032      74.790000
Name: price_per_unit, Length: 224, dtype: float64

In [11]:
df

,order_time,qty,total_price,SKU,price_per_unit
0,2025-05-26 06:30:18,1,312.68,ga10333,312.680
1,2024-09-23 12:21:21,2,450.37,ga10161,225.185
2,2024-10-09 12:18:14,1,224.57,ga10161,224.570
3,2024-09-19 12:29:59,1,205.70,ga10312,205.700
4,2025-06-18 14:59:13,1,298.15,ga10313,298.150
...,...,...,...,...,...
6352,2024-08-13 08:42:00,3,112.50,sw10004,37.500
6353,2024-07-15 11:25:24,1,50.67,sw100171,50.670
6354,2024-08-13 08:42:00,2,75.00,sw10017,37.500
6355,2024-06-17 07:26:08,1,50.42,sw100173,50.420


In [12]:
df["date"] = pd.to_datetime(df["order_time"]).dt.floor("D")

In [21]:

# 1) Aggregate existing events per SKU per day
daily = (
    df
    .groupby(["SKU", "date"], as_index=False)
    .agg(
        qty=("qty", "sum"),
        price_per_unit=("price_per_unit", "mean"),
    )
)

In [22]:
daily

,SKU,date,qty,price_per_unit
0,9101dx,2024-01-26,2,1522.660000
1,9101dx,2024-01-29,2,1449.580000
2,9101dx,2024-01-30,1,1449.580000
3,9101dx,2024-02-05,1,1562.500000
4,9101dx,2024-02-06,2,1294.260000
...,...,...,...,...
5262,sz21031,2024-05-18,2,74.790000
5263,sz21031,2024-11-18,2,113.445000
5264,sz21031,2025-08-08,3,136.126667
5265,sz21032,2024-06-17,2,74.790000


In [23]:
# 2) Build full SKU x date grid
all_skus = daily["SKU"].unique()
full_dates = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")

full_index = pd.MultiIndex.from_product(
    [all_skus, full_dates],
    names=["SKU", "date"]
)

In [30]:
daily_full = (
    daily
    .set_index(["SKU", "date"])
    .reindex(full_index)
    .reset_index()
)

In [31]:
daily_full["qty"] = daily_full["qty"].fillna(0)

In [33]:
daily_full["price_per_unit"] = daily_full.apply(lambda row: mean_price_per_sku[row["SKU"]] if np.isnan(row["price_per_unit"]) else row["price_per_unit"], axis=1)

In [36]:
agg = daily_full

In [37]:
agg

,SKU,date,qty,price_per_unit
0,9101dx,2024-01-26,2.0,1522.660000
1,9101dx,2024-01-27,0.0,1295.228583
2,9101dx,2024-01-28,0.0,1295.228583
3,9101dx,2024-01-29,2.0,1449.580000
4,9101dx,2024-01-30,1.0,1449.580000
...,...,...,...,...
149851,sz21032,2025-11-20,0.0,74.790000
149852,sz21032,2025-11-21,0.0,74.790000
149853,sz21032,2025-11-22,0.0,74.790000
149854,sz21032,2025-11-23,0.0,74.790000


In [38]:
# WEATHER DATA

In [39]:
from geopy.geocoders import Nominatim
from datetime import datetime
from meteostat import Point, Daily

geolocator = Nominatim(user_agent="eleo-mind")
location = geolocator.geocode("Stuttgart, Germany")
geo_point = Point(location.latitude, location.longitude)

In [40]:
date_min = agg["date"].min()
date_max = agg["date"].max()

In [42]:
daily_weather = Daily(geo_point, date_min, date_max).fetch()

In [43]:
relevant_weather = daily_weather[["tavg", "prcp", "tsun"]]

In [49]:
agg

,date,SKU,qty,price_per_unit,tavg,prcp,tsun
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0
1,2024-01-27,9101dx,0.0,1295.228583,3.0,0.0,510.0
2,2024-01-28,9101dx,0.0,1295.228583,3.7,0.0,516.0
3,2024-01-29,9101dx,2.0,1449.580000,5.5,0.0,492.0
4,2024-01-30,9101dx,1.0,1449.580000,6.1,0.0,192.0
...,...,...,...,...,...,...,...
149851,2025-11-20,sz21032,0.0,74.790000,2.3,0.0,59.0
149852,2025-11-21,sz21032,0.0,74.790000,-1.0,0.0,136.0
149853,2025-11-22,sz21032,0.0,74.790000,-4.2,0.0,243.0
149854,2025-11-23,sz21032,0.0,74.790000,-2.6,2.5,155.0


In [45]:
agg = agg.set_index("date")
agg = agg.join(relevant_weather, how="left")
agg = agg.reset_index()

In [46]:
agg.isna().sum()

date              0
SKU               0
qty               0
price_per_unit    0
tavg              0
prcp              0
tsun              0
dtype: int64

In [47]:
agg

,date,SKU,qty,price_per_unit,tavg,prcp,tsun
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0
1,2024-01-27,9101dx,0.0,1295.228583,3.0,0.0,510.0
2,2024-01-28,9101dx,0.0,1295.228583,3.7,0.0,516.0
3,2024-01-29,9101dx,2.0,1449.580000,5.5,0.0,492.0
4,2024-01-30,9101dx,1.0,1449.580000,6.1,0.0,192.0
...,...,...,...,...,...,...,...
149851,2025-11-20,sz21032,0.0,74.790000,2.3,0.0,59.0
149852,2025-11-21,sz21032,0.0,74.790000,-1.0,0.0,136.0
149853,2025-11-22,sz21032,0.0,74.790000,-4.2,0.0,243.0
149854,2025-11-23,sz21032,0.0,74.790000,-2.6,2.5,155.0


In [50]:
aktionen_df = pd.read_csv("/Users/rhombus19/projects/eleo/marketingaktionen_2024_2025_aktionen_sku.csv")

In [51]:
sparten_mapping = pd.read_csv("/Users/rhombus19/projects/eleo/marketingaktionen_2024_2025_produktsparte_sku_mapping.csv")

In [52]:
aktionen_df

,Monat,Jahr,Zeitraum_Beginn,Zeitraum_Ende,Produktsparte,SKU,Aktionstext,Einschraenkung_Text
0,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9101ub,50 € ab 500 €\n100 € ab 1000 €,NaN
1,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9101fz,50 € ab 500 €\n100 € ab 1000 €,NaN
2,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9101dx,50 € ab 500 €\n100 € ab 1000 €,NaN
3,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9101sf,50 € ab 500 €\n100 € ab 1000 €,NaN
4,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9102ub,50 € ab 500 €\n100 € ab 1000 €,NaN
...,...,...,...,...,...,...,...,...
5417,November,2025,2025-11-13,2025-11-30,Produktsparte 3,SZ21022,20 % auf alles,NaN
5418,November,2025,2025-11-13,2025-11-30,Produktsparte 3,SZ2102sf,20 % auf alles,NaN
5419,November,2025,2025-11-13,2025-11-30,Produktsparte 3,SZ21031,20 % auf alles,NaN
5420,November,2025,2025-11-13,2025-11-30,Produktsparte 3,SZ21032,20 % auf alles,NaN


In [53]:
aktionen_df["Aktionstext"].unique().tolist()

['50 €  ab 500 €\n100 € ab 1000 €',
 '24 % auf alles',
 '15 % auf alles',
 '10% auf alles',
 '10 % auf ausgewählte Artikel',
 '15 % auf ausgewählte Artikel',
 '10 % auf alles',
 '15 % auf ausgewählte Artikel, 10 % auf alles andere ',
 '15 % ausgewählte Artikel',
 '20 % auf alles',
 'Montageservice geschenkt, Versandkostenfreie Lieferung',
 'Versandkostenfrei',
 '12 % auf alles',
 '20 % alles',
 '15% auf alles',
 '12 % auf ausgewählte Artikel',
 '3 Artikel kaufen - günstigsten gratis',
 'KEIN RABATT']

In [54]:
rabbat_mapping = {'50 €  ab 500 €\n100 € ab 1000 €': 0.10,
'24 % auf alles': 0.24,
'15 % auf alles': 0.15,
'10% auf alles': 0.10,
'10 % auf ausgewählte Artikel': 0.10,
'15 % auf ausgewählte Artikel': 0.15,
'10 % auf alles': 0.1,
'15 % auf ausgewählte Artikel, 10 % auf alles andere ': 0.15,
'15 % ausgewählte Artikel': 0.15,
'20 % auf alles': 0.20,
'Montageservice geschenkt, Versandkostenfreie Lieferung': 0.08,
'Versandkostenfrei': 0.05,
'12 % auf alles': 0.12,
'20 % alles': 0.20,
'15% auf alles': 0.15,
'12 % auf ausgewählte Artikel': 0.12,
'3 Artikel kaufen - günstigsten gratis': 0.33,
'KEIN RABATT': 0.0}

In [55]:
aktionen_df

,Monat,Jahr,Zeitraum_Beginn,Zeitraum_Ende,Produktsparte,SKU,Aktionstext,Einschraenkung_Text
0,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9101ub,50 € ab 500 €\n100 € ab 1000 €,NaN
1,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9101fz,50 € ab 500 €\n100 € ab 1000 €,NaN
2,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9101dx,50 € ab 500 €\n100 € ab 1000 €,NaN
3,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9101sf,50 € ab 500 €\n100 € ab 1000 €,NaN
4,Januar,2024,2024-01-15,2024-01-15,Produktsparte 1,9102ub,50 € ab 500 €\n100 € ab 1000 €,NaN
...,...,...,...,...,...,...,...,...
5417,November,2025,2025-11-13,2025-11-30,Produktsparte 3,SZ21022,20 % auf alles,NaN
5418,November,2025,2025-11-13,2025-11-30,Produktsparte 3,SZ2102sf,20 % auf alles,NaN
5419,November,2025,2025-11-13,2025-11-30,Produktsparte 3,SZ21031,20 % auf alles,NaN
5420,November,2025,2025-11-13,2025-11-30,Produktsparte 3,SZ21032,20 % auf alles,NaN


In [56]:
aktionen_df["Zeitraum_Beginn"] = pd.to_datetime(aktionen_df["Zeitraum_Beginn"])
aktionen_df["Zeitraum_Ende"] = pd.to_datetime(aktionen_df["Zeitraum_Ende"])

In [57]:
def is_sale_active(date, sku):
    active_sales = aktionen_df[
        (aktionen_df["Zeitraum_Beginn"] <= date) &
        (aktionen_df["Zeitraum_Ende"] >= date) &
        (aktionen_df["SKU"] == sku)
    ]
    return not active_sales.empty

In [58]:
def get_sale_percentage(date, sku):
    active_sales = aktionen_df[
        (aktionen_df["Zeitraum_Beginn"] <= date) &
        (aktionen_df["Zeitraum_Ende"] >= date) &
        (aktionen_df["SKU"] == sku)
    ]
    if not active_sales.empty:
        rabatte = active_sales["Aktionstext"].map(rabbat_mapping)
        return rabatte.max()
    return 0.0

In [59]:
agg["sale_active"] = agg.progress_apply(lambda row: is_sale_active(row["date"], row["SKU"]), axis=1)

100%|██████████| 149856/149856 [00:47<00:00, 3159.37it/s]


In [60]:
agg["sale_percent"] = agg.progress_apply(lambda row: get_sale_percentage(row["date"], row["SKU"]), axis=1)

100%|██████████| 149856/149856 [00:53<00:00, 2807.05it/s]


In [61]:
agg

,date,SKU,qty,price_per_unit,tavg,prcp,tsun,sale_active,sale_percent
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0,True,0.24
1,2024-01-27,9101dx,0.0,1295.228583,3.0,0.0,510.0,True,0.24
2,2024-01-28,9101dx,0.0,1295.228583,3.7,0.0,516.0,True,0.24
3,2024-01-29,9101dx,2.0,1449.580000,5.5,0.0,492.0,True,0.24
4,2024-01-30,9101dx,1.0,1449.580000,6.1,0.0,192.0,True,0.24
...,...,...,...,...,...,...,...,...,...
149851,2025-11-20,sz21032,0.0,74.790000,2.3,0.0,59.0,False,0.00
149852,2025-11-21,sz21032,0.0,74.790000,-1.0,0.0,136.0,False,0.00
149853,2025-11-22,sz21032,0.0,74.790000,-4.2,0.0,243.0,False,0.00
149854,2025-11-23,sz21032,0.0,74.790000,-2.6,2.5,155.0,False,0.00


In [62]:
agg["mmdd"] = agg["date"].dt.strftime("%m-%d")
agg["day_of_week"] = agg["date"].dt.dayofweek
agg["month"] = agg["date"].dt.month

In [63]:
import feiertage
def is_holiday(date):
    return date.to_pydatetime().date() in feiertage.Holidays(state="BY", year=date.year, regional=True, school_free=True).get_holidays_list()

In [64]:
agg["is_holiday"] = agg["date"].apply(is_holiday)

In [84]:
agg

,date,SKU,qty,price_per_unit,tavg,prcp,tsun,sale_active,sale_percent,mmdd,day_of_week,month,is_holiday
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0,True,0.24,01-26,4,1,False
1,2024-01-27,9101dx,0.0,1295.228583,3.0,0.0,510.0,True,0.24,01-27,5,1,False
2,2024-01-28,9101dx,0.0,1295.228583,3.7,0.0,516.0,True,0.24,01-28,6,1,False
3,2024-01-29,9101dx,2.0,1449.580000,5.5,0.0,492.0,True,0.24,01-29,0,1,False
4,2024-01-30,9101dx,1.0,1449.580000,6.1,0.0,192.0,True,0.24,01-30,1,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149851,2025-11-20,sz21032,0.0,74.790000,2.3,0.0,59.0,False,0.00,11-20,3,11,False
149852,2025-11-21,sz21032,0.0,74.790000,-1.0,0.0,136.0,False,0.00,11-21,4,11,False
149853,2025-11-22,sz21032,0.0,74.790000,-4.2,0.0,243.0,False,0.00,11-22,5,11,False
149854,2025-11-23,sz21032,0.0,74.790000,-2.6,2.5,155.0,False,0.00,11-23,6,11,False


In [85]:
agg.to_csv("/Users/rhombus19/projects/eleo/forecast_dataset.csv", index=False)

In [66]:
# Get weather forcast for the future

In [67]:
from dateutil.relativedelta import relativedelta
from functools import cache

@cache
def get_avg_weather_per_mmdd(geo_point, years_avg=10):
    hist_weather = Daily(geo_point, datetime.now() - relativedelta(years=years_avg), datetime.now()).fetch()
    hist_weather = hist_weather.reset_index()
    hist_weather = hist_weather.dropna(subset=["tavg"])

    hist_weather["mmdd"] = hist_weather["time"].dt.strftime("%m-%d")

    # Take median per calendar day over all years
    climatology = (
        hist_weather
        .groupby("mmdd")[["tavg", "prcp", "snow", "tsun"]]
        .median()
        .reset_index()
    )
    return climatology

In [68]:
new_dates = pd.date_range(start=agg["date"].max(), end=agg["date"].max() + pd.Timedelta(days=100))

In [69]:
new_dates

DatetimeIndex(['2025-11-24', '2025-11-25', '2025-11-26', '2025-11-27',
               '2025-11-28', '2025-11-29', '2025-11-30', '2025-12-01',
               '2025-12-02', '2025-12-03',
               ...
               '2026-02-23', '2026-02-24', '2026-02-25', '2026-02-26',
               '2026-02-27', '2026-02-28', '2026-03-01', '2026-03-02',
               '2026-03-03', '2026-03-04'],
              dtype='datetime64[ns]', length=101, freq='D')

In [70]:
daily_weather = Daily(geo_point, new_dates.min(), new_dates.max()).fetch()

In [71]:
daily_weather[["tavg", "prcp", "tsun"]]

,tavg,prcp,tsun
time,,,
2025-11-24,4.1,5.8,0.0
2025-11-25,3.6,7.1,17.0
2025-11-26,3.3,2.2,0.0
2025-11-27,1.9,0.0,194.0
2025-11-28,2.9,0.6,88.0
2025-11-29,6.0,2.4,88.0
2025-11-30,5.2,1.6,74.0
2025-12-01,3.8,0.0,88.0
2025-12-02,5.1,0.0,115.0


In [72]:
daily_weather = Daily(geo_point, new_dates.min(), new_dates.max()).fetch()

weather = daily_weather[["tavg", "prcp", "tsun"]].copy()
weather.index.name = "date"          # was "time"

# 2) Turn new_dates into a DataFrame
future_dates = pd.DataFrame({"date": new_dates})

# 3) Merge on 'date'
future_weather = future_dates.merge(
    weather.reset_index(),  # brings 'date' back as a column
    on="date",
    how="left"
)

climate_avg_weather = get_avg_weather_per_mmdd(geo_point)

climate_idx = climate_avg_weather.set_index('mmdd')

# 2) Derive mmdd from the full date in future_weather
mmdd = pd.to_datetime(future_weather['date']).dt.strftime('%m-%d')

# 3) For each weather variable, fill NaNs in future_weather
for col in ['tavg', 'prcp', 'tsun']:
    future_weather[col] = future_weather[col].fillna(mmdd.map(climate_idx[col]))

In [73]:
future_weather

,date,tavg,prcp,tsun
0,2025-11-24,4.1,5.8,0.0
1,2025-11-25,3.6,7.1,17.0
2,2025-11-26,3.3,2.2,0.0
3,2025-11-27,1.9,0.0,194.0
4,2025-11-28,2.9,0.6,88.0
...,...,...,...,...
96,2026-02-28,3.85,0.0,333.0
97,2026-03-01,4.2,0.05,174.0
98,2026-03-02,5.3,0.0,252.0
99,2026-03-03,4.55,0.0,282.0


In [74]:
future_dates = future_weather.copy()

In [75]:
future_dates["is_holiday"] = future_dates["date"].apply(is_holiday)

In [76]:
sku_df = pd.DataFrame({"SKU": all_skus})

future_df = future_dates.merge(sku_df, how="cross")

In [77]:
future_df

,date,tavg,prcp,tsun,is_holiday,SKU
0,2025-11-24,4.1,5.8,0.0,False,9101dx
1,2025-11-24,4.1,5.8,0.0,False,9101fz
2,2025-11-24,4.1,5.8,0.0,False,9101sf
3,2025-11-24,4.1,5.8,0.0,False,9101ub
4,2025-11-24,4.1,5.8,0.0,False,9102dx
...,...,...,...,...,...,...
22619,2026-03-04,6.25,0.05,234.0,False,sz21011
22620,2026-03-04,6.25,0.05,234.0,False,sz21012
22621,2026-03-04,6.25,0.05,234.0,False,sz21022
22622,2026-03-04,6.25,0.05,234.0,False,sz21031


In [78]:
future_df["sale_percent"] = future_df.apply(lambda row: get_sale_percentage(row["date"], row["SKU"]), axis=1)

In [79]:
future_df

,date,tavg,prcp,tsun,is_holiday,SKU,sale_percent
0,2025-11-24,4.1,5.8,0.0,False,9101dx,0.2
1,2025-11-24,4.1,5.8,0.0,False,9101fz,0.2
2,2025-11-24,4.1,5.8,0.0,False,9101sf,0.2
3,2025-11-24,4.1,5.8,0.0,False,9101ub,0.2
4,2025-11-24,4.1,5.8,0.0,False,9102dx,0.2
...,...,...,...,...,...,...,...
22619,2026-03-04,6.25,0.05,234.0,False,sz21011,0.0
22620,2026-03-04,6.25,0.05,234.0,False,sz21012,0.0
22621,2026-03-04,6.25,0.05,234.0,False,sz21022,0.0
22622,2026-03-04,6.25,0.05,234.0,False,sz21031,0.0


In [80]:
future_df["month"] = future_df["date"].dt.month
future_df["day_of_week"] = future_df["date"].dt.dayofweek
future_df["mmdd"] = future_df["date"].dt.strftime("%m-%d")

In [81]:
future_df["price_per_unit"] = future_df["SKU"].map(mean_price_per_sku)

In [82]:
future_df

,date,tavg,prcp,tsun,is_holiday,SKU,sale_percent,month,day_of_week,mmdd,price_per_unit
0,2025-11-24,4.1,5.8,0.0,False,9101dx,0.2,11,0,11-24,1295.228583
1,2025-11-24,4.1,5.8,0.0,False,9101fz,0.2,11,0,11-24,1119.547500
2,2025-11-24,4.1,5.8,0.0,False,9101sf,0.2,11,0,11-24,1468.438148
3,2025-11-24,4.1,5.8,0.0,False,9101ub,0.2,11,0,11-24,864.650000
4,2025-11-24,4.1,5.8,0.0,False,9102dx,0.2,11,0,11-24,1937.436548
...,...,...,...,...,...,...,...,...,...,...,...
22619,2026-03-04,6.25,0.05,234.0,False,sz21011,0.0,3,2,03-04,74.790000
22620,2026-03-04,6.25,0.05,234.0,False,sz21012,0.0,3,2,03-04,74.375417
22621,2026-03-04,6.25,0.05,234.0,False,sz21022,0.0,3,2,03-04,59.830000
22622,2026-03-04,6.25,0.05,234.0,False,sz21031,0.0,3,2,03-04,108.120556


In [83]:
future_df.to_csv("/Users/rhombus19/projects/eleo/future_features_dataset.csv", index=False)